# **05_Logistic_Regression.ipynb**


Objective:
To train and assess Logistic Regression models on Elliptic and Ethereum datasets, using the provided train/test partitions.

Inputs:

*   elliptic_train.csv
*   elliptic_test.csv
*   ethereum_train.csv
*   ethereum_test.csv

Outputs:
*   models/lr_elliptic.joblib
*   models/lr_ethereum.joblib
*   models/scaler_elliptic.joblib
*   models/scaler_ethereum.joblib
*   models/lr_results.json









In [14]:
import os
import json
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,f1_score, roc_auc_score, confusion_matrix)

In [15]:
def evaluate(model, X_test, y_test):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, prob),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    }

In [16]:
# ELLIPTIC

ell_train = pd.read_csv(
    "../train_test_data/elliptic_train.csv"
)

ell_test = pd.read_csv(
    "../train_test_data/elliptic_test.csv"
)

feat_cols = [
    c for c in ell_train.columns
    if c.startswith("feat_")
]

print("Elliptic train:", ell_train.shape)
print("Elliptic test:", ell_test.shape)
print("Number of features:", len(feat_cols))

Elliptic train: (29894, 168)
Elliptic test: (16670, 168)
Number of features: 166


In [17]:
X_train, y_train = ell_train[feat_cols], ell_train["label"]
X_test, y_test = ell_test[feat_cols], ell_test["label"]

In [18]:
scaler_ell = StandardScaler()
X_train_s = scaler_ell.fit_transform(X_train)
X_test_s = scaler_ell.transform(X_test)

In [19]:
lr_ell = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
lr_ell.fit(X_train_s, y_train)
results["elliptic_lr"] = evaluate(lr_ell, X_test_s, y_test)


In [20]:
# ETHEREUM

eth_train = pd.read_csv("../train_test_data/ethereum_train.csv")
eth_test = pd.read_csv("../train_test_data/ethereum_test.csv")
feat_cols_eth = [c for c in eth_train.columns if c not in ["Address", "FLAG"]]

In [21]:
X_train_e, y_train_e = eth_train[feat_cols_eth], eth_train["FLAG"]
X_test_e, y_test_e = eth_test[feat_cols_eth], eth_test["FLAG"]


In [22]:
scaler_eth = StandardScaler()
X_train_es = scaler_eth.fit_transform(X_train_e)
X_test_es = scaler_eth.transform(X_test_e)

In [23]:
lr_eth = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
lr_eth.fit(X_train_es, y_train_e)
results["ethereum_lr"] = evaluate(lr_eth, X_test_es, y_test_e)

In [24]:
# Display Logistic Regression results

for name, r in results.items():
    print(f"\n--- {name} ---")

    for k, v in r.items():
        if k != "confusion_matrix":
            print(f"{k:>10}: {v:.4f}")

    print("confusion matrix:", r["confusion_matrix"])


--- elliptic_lr ---
  accuracy: 0.7375
 precision: 0.1833
    recall: 0.8800
        f1: 0.3034
   roc_auc: 0.8822
confusion matrix: [[11341, 4246], [130, 953]]

--- ethereum_lr ---
  accuracy: 0.6599
 precision: 0.3771
    recall: 0.8165
        f1: 0.5159
   roc_auc: 0.8834
confusion matrix: [[940, 588], [80, 356]]


In [26]:
# Save Logistic Regression models, scalers and results

MODELS_PATH = "../models"
os.makedirs(MODELS_PATH, exist_ok=True)

joblib.dump(
    lr_ell,
    os.path.join(MODELS_PATH, "lr_elliptic.joblib")
)

joblib.dump(
    scaler_ell,
    os.path.join(MODELS_PATH, "scaler_elliptic.joblib")
)

joblib.dump(
    lr_eth,
    os.path.join(MODELS_PATH, "lr_ethereum.joblib")
)

joblib.dump(
    scaler_eth,
    os.path.join(MODELS_PATH, "scaler_ethereum.joblib")
)

with open(
    os.path.join(MODELS_PATH, "lr_results.json"),
    "w"
) as f:
    json.dump(results, f, indent=2)

print("Logistic Regression models and results saved successfully.")

Logistic Regression models and results saved successfully.


In [27]:
print("\nFiles saved in models:")

for file in sorted(os.listdir(MODELS_PATH)):
    print(file)


Files saved in models:
lr_elliptic.joblib
lr_ethereum.joblib
lr_results.json
rf_elliptic.joblib
rf_ethereum.joblib
rf_results.json
scaler_elliptic.joblib
scaler_ethereum.joblib
